In [38]:
import json
import shutil
import zipfile
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

## Collect Unique Product IDs

In [39]:
label_path = Path("fashion_compatibility_prediction.txt")
product_ids = set()

with open(label_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()

        # Skip label (first element)
        product_ids.update(parts[1:])

print("Unique products:", len(product_ids))

Unique products: 18604


## Build Original Outfit Table

In [40]:
records = []

with open("fashion_compatibility_prediction.txt", "r") as f:
    for i, line in enumerate(f):
        parts = line.strip().split()
        records.append({
            "outfit_id": f"O{i+1:04d}",
            "label": int(parts[0]),
            "products": parts[1:]
        })

df = pd.DataFrame(records)
df

,outfit_id,label,products
0,O0001,1,"[119704139_1, 119704139_2, 119704139_3, 119704..."
1,O0002,1,"[119314458_1, 119314458_2, 119314458_3, 119314..."
2,O0003,1,"[147361785_1, 147361785_2, 147361785_3, 147361..."
3,O0004,1,"[148015877_1, 148015877_2, 148015877_3, 148015..."
4,O0005,1,"[192637078_1, 192637078_2, 192637078_3, 192637..."
...,...,...,...
7071,O7072,0,"[184041913_1, 209222683_1, 185007152_3, 207694..."
7072,O7073,0,"[209709393_3, 212653508_4, 189949776_6, 166128..."
7073,O7074,0,"[109684687_6, 104980596_1, 179458125_4, 129142..."
7074,O7075,0,"[215502815_1, 207577310_1, 132234026_1, 127969..."


## Define Garment Categories

In [41]:
target_folders = {
    "outwear": "OUTERWEAR",
    "top": "TOP",
    "pants": "BOTTOM",
    "skirt": "BOTTOM",
    "shoes": "FOOTWEAR"
}

category_order = {
    "OUTERWEAR": 0,
    "TOP": 1,
    "BOTTOM": 2,
    "FOOTWEAR": 3
}

## Extract Matching Images (from target_folders only)

In [42]:
zip_path = "Re-PolyVore.zip"

found_ids = set()
product_category = {}
product_image_path = {}

with zipfile.ZipFile(zip_path, "r") as z:

    image_paths = [
        name for name in z.namelist()
        if name.lower().endswith(".jpg")
        and Path(name).parent.name.lower() in target_folders
    ]

    for path in tqdm(image_paths):

        folder = Path(path).parent.name.lower()
        product_id = Path(path).stem

        if product_id in product_ids:
            found_ids.add(product_id)
            product_category[product_id] = target_folders[folder]
            product_image_path[product_id] = path

print("Required products:", len(product_ids))
print("Found images:", len(found_ids))
print("Missing images:", len(product_ids - found_ids))

100%|██████████| 63965/63965 [00:00<00:00, 268909.03it/s]

Required products: 18604
Found images: 4517
Missing images: 14087


## Clean and Sort Products

In [43]:
def clean_and_sort_products(products):
    # Remove products without images
    valid_products = [pid for pid in products if pid in found_ids]

    # Sort products by category
    return sorted(valid_products, key=lambda pid: category_order[product_category[pid]])


df_clean = df.copy()
df_clean["products"] = df_clean["products"].apply(clean_and_sort_products)

# Keep outfits with at least 2 products
df_clean = df_clean[
    df_clean["products"].apply(len) >= 2
].reset_index(drop=True)

print("Original outfits:", len(df))
print("Remaining outfits:", len(df_clean))
print("Removed outfits:", len(df) - len(df_clean))
print(df_clean["label"].value_counts())

Original outfits: 7076
Remaining outfits: 3137
Removed outfits: 3939
label
0    1744
1    1393
Name: count, dtype: int64


## Remove Duplicate Categories

In [44]:
def has_unique_categories(products):
    categories = [product_category[pid] for pid in products]
    return len(categories) == len(set(categories))


df_filtered = df_clean[
    df_clean["products"].apply(has_unique_categories)
].reset_index(drop=True)

print("Before filtering:", len(df_clean))
print("Remaining outfits:", len(df_filtered))
print("Removed outfits:", len(df_clean) - len(df_filtered))
print(df_filtered["label"].value_counts())

Before filtering: 3137
Remaining outfits: 2377
Removed outfits: 760
label
1    1274
0    1103
Name: count, dtype: int64


## Export Cleaned Dataset

In [45]:
# Collect product IDs used in the final dataset
df_final = df_filtered.copy()
final_product_ids = {pid for products in df_final["products"] for pid in products}

# 1. Save cleaned outfits
df_save = df_final[["outfit_id", "label", "products"]].copy()
df_save["products"] = df_save["products"].apply(json.dumps)
df_save.to_csv("clean_outfits_compatibility_prediction.csv", index=False)

# 2. Save category mapping (only final products)
final_product_category = {pid: product_category[pid] for pid in final_product_ids}

with open("product_category.json", "w", encoding="utf-8") as f:
    json.dump(final_product_category, f, indent=2)

# 3. Extract images (only final products)
output_dir = Path("polyvore_selected_images")
output_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    for pid in tqdm(sorted(final_product_ids)):
        image_path = product_image_path[pid]
        output_path = output_dir / f"{pid}.jpg"

        with z.open(image_path) as source:
            with open(output_path, "wb") as target:
                shutil.copyfileobj(source, target)

shutil.make_archive(
    base_name="polyvore_selected_images", 
    format="zip", 
    root_dir=".", 
    base_dir="polyvore_selected_images"
)

shutil.rmtree(output_dir)

print("Saved outfits:", len(df_final))
print("Saved product categories:", len(final_product_category))
print("Saved images:", len(final_product_ids))

100%|██████████| 3789/3789 [00:03<00:00, 1230.24it/s]


Saved outfits: 2377
Saved product categories: 3789
Saved images: 3789
